# Teste básico — todos os protocolos de uma vez

Roda **todos** os protocolos registrados sobre uma amostra do GSM8K e mostra a
tabela comparativa (acurácia × latência × custo). Inclui o **debate clássico**
("society of minds", voto majoritário) junto com Single Minion, Single Agent,
Minions e Mixture-of-Agents.

Por padrão usa o backend **mock** (roda em qualquer máquina, sem GPU). Para
rodar os modelos reais na GPU, troque `MULTIAGENT_BACKEND` para `"hf"` na
célula de configuração abaixo.


In [ ]:
import os
import sys

# ── Configuração do teste ─────────────────────────────────────────────
# "mock": roda em qualquer máquina (sem GPU, sem baixar modelos).
# "hf"  : roda os modelos reais (Qwen2.5) na GPU.
os.environ["MULTIAGENT_BACKEND"] = "hf"   # troque para "hf" na A100
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # (modo hf) escolha a GPU

N_SAMPLES = 20   # nº de perguntas do GSM8K (pequeno p/ um teste rápido)
SEED = 42

# Permite importar config/src com o notebook na raiz do projeto.
sys.path.insert(0, os.path.abspath("."))

## Protocolos registrados


In [ ]:
from src.protocols import available

print("Protocolos registrados:", available())

## Rodar todo mundo de uma vez

Equivale, na linha de comando, a `python scripts/run_experiment.py --all`.


In [ ]:
from dataclasses import replace

import config as cfg
from src.protocols import available
from src.runner import run_experiment

# Ordem didática: piso -> teto -> meio -> debate (só os que existem).
ordem = ["single_minion", "single_agent", "minions", "mixture_of_agents", "debate"]
protocolos = tuple(p for p in ordem if p in available())

exp = replace(cfg.EXPERIMENT, n_samples=N_SAMPLES, seed=SEED, protocols=protocolos)
aggs = run_experiment(exp)

## Tabela comparativa


In [ ]:
import pandas as pd

df = pd.DataFrame([a.as_row() for a in aggs]).set_index("protocol")

# Versão amigável para leitura (sem depender de jinja2/.style):
view = df.copy()
view["accuracy"] = (view["accuracy"] * 100).round(1)            # %
view["master_usage_rate"] = (view["master_usage_rate"] * 100).round(1)  # %
view = view.round({
    "avg_latency_s": 3, "avg_total_tokens": 0,
    "avg_master_tokens": 0, "avg_model_calls": 2,
})
view = view.rename(columns={"accuracy": "accuracy_%", "master_usage_rate": "master_usage_%"})
view

## Gráficos rápidos


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["accuracy"].plot.bar(ax=axes[0], title="Acurácia", color="#4C9F70")
df["avg_latency_s"].plot.bar(ax=axes[1], title="Latência média (s)", color="#E07A5F")
df["master_usage_rate"].plot.bar(ax=axes[2], title="Uso do modelo grande", color="#3D5A80")
for ax in axes:
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()